In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, json, warnings, random
from datetime import datetime, timedelta
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import pickle

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUTPUT_DIR = Path("./aml_pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Imports complete.")

# ============================================================
# STAGE 1 — REALISTIC SYNTHETIC DATA GENERATION
# ============================================================
# %% [markdown]
# ## Stage 1: Realistic Synthetic Bank Data Generation
#
# We mimic the data structures present in real bank AML systems:
# - **Customers** : KYC profile, risk rating, PEP/sanctions flags
# - **Accounts**  : Account type, balance, dormancy, linked entities
# - **Transactions**: Payment rails, amounts, counterparties, geographies
# - **Alerts**    : Rule engine outputs with typology tags
# - **Cases**     : Investigator decisions rolled up from alerts
# - **STRs**      : Ground truth — was a Suspicious Transaction Report filed?

# %% — CONFIGURATION
N_CUSTOMERS   = 8_000
N_ACCOUNTS    = 12_000
N_TXNS        = 120_000
START_DATE    = datetime(2022, 1, 1)
END_DATE      = datetime(2024, 6, 30)
start_ts, end_ts = START_DATE.timestamp(), END_DATE.timestamp()

print(f"Generating synthetic data: {N_CUSTOMERS:,} customers | {N_ACCOUNTS:,} accounts | {N_TXNS:,} transactions")

# ── HELPER FUNCTIONS ──────────────────────────────────────────
def rand_dates(n, lo=start_ts, hi=end_ts):
    return [datetime.fromtimestamp(t) for t in np.random.uniform(lo, hi, n)]

def lognorm(mean_val, sigma, size):
    """Return lognormal samples centred on mean_val."""
    mu = np.log(mean_val) - 0.5 * sigma**2
    return np.round(np.random.lognormal(mu, sigma, size), 2)

# ── 1A. CUSTOMERS ─────────────────────────────────────────────
COUNTRIES   = ["IN", "US", "AE", "SG", "GB", "CN", "MY", "KY", "VG", "NG"]
COUNTRY_W   = [0.40, 0.15, 0.10, 0.08, 0.07, 0.06, 0.04, 0.03, 0.04, 0.03]
OCCUPATIONS = ["SALARIED", "SELF_EMPLOYED", "BUSINESS_OWNER",
                "RETIRED", "STUDENT", "POLITICIAN", "SHELL_CO"]
OCC_W       = [0.38, 0.22, 0.18, 0.08, 0.05, 0.05, 0.04]

cust_ids = [f"CUST{i:07d}" for i in range(1, N_CUSTOMERS + 1)]

# Risk rating: 1-10 (1=lowest, 10=highest)
# Modelled as mixture: most customers are low risk
base_risk = np.random.choice(np.arange(1, 11),
                              p=[0.20, 0.20, 0.15, 0.12, 0.10,
                                 0.08, 0.06, 0.04, 0.03, 0.02],
                              size=N_CUSTOMERS)

occupation = np.random.choice(OCCUPATIONS, p=OCC_W, size=N_CUSTOMERS)
country    = np.random.choice(COUNTRIES,   p=COUNTRY_W, size=N_CUSTOMERS)

# PEP: higher if politician or offshore jurisdiction
pep_prob = np.where(occupation == "POLITICIAN", 0.90,
           np.where(np.isin(country, ["KY", "VG"]), 0.25, 0.02))
pep_flag = np.random.rand(N_CUSTOMERS) < pep_prob

# Sanctions hit simulation
sanction_prob = np.where(base_risk > 8, 0.08,
                np.where(base_risk > 6, 0.02, 0.003))
sanction_flag = np.random.rand(N_CUSTOMERS) < sanction_prob

# Adverse media flag
adverse_media = np.random.rand(N_CUSTOMERS) < (base_risk / 100 + 0.01)

# KYC completeness (%) — inversely correlated with risk
kyc_completeness = np.clip(
    np.random.normal(loc=100 - base_risk * 4, scale=8, size=N_CUSTOMERS), 30, 100
).round(1)

# Days since last KYC refresh
days_since_kyc = np.clip(
    np.random.exponential(scale=180, size=N_CUSTOMERS).astype(int), 1, 1800
)

customers_df = pd.DataFrame({
    "CUSTOMER_ID"       : cust_ids,
    "NATIONALITY"       : country,
    "OCCUPATION"        : occupation,
    "INCOME_BRACKET"    : np.random.choice(["0-50k","50k-100k","100k-250k","250k+"],
                                            p=[0.35,0.30,0.22,0.13], size=N_CUSTOMERS),
    "CUSTOMER_RISK_RATING": base_risk,        # 1-10
    "PEP_FLAG"          : pep_flag.astype(int),
    "SANCTION_HIT"      : sanction_flag.astype(int),
    "ADVERSE_MEDIA_FLAG": adverse_media.astype(int),
    "KYC_COMPLETENESS_PCT": kyc_completeness,
    "DAYS_SINCE_KYC"    : days_since_kyc,
    "ONBOARDING_CHANNEL": np.random.choice(["BRANCH","MOBILE","WEB","THIRD_PARTY"],
                                            p=[0.30,0.35,0.25,0.10], size=N_CUSTOMERS),
    "IS_ACTIVE"         : np.random.choice([1,0], p=[0.94,0.06], size=N_CUSTOMERS),
})

print(f"  Customers: {customers_df.shape}")

# ── 1B. ACCOUNTS ──────────────────────────────────────────────
acct_ids = [f"ACCT{100000000 + i}" for i in range(1, N_ACCOUNTS + 1)]
acct_cust = np.random.choice(cust_ids, size=N_ACCOUNTS)

# Balance modelled by account type
acct_type = np.random.choice(
    ["SAVINGS","CURRENT","SALARY","NRE","NRO","TRUST","SHELL"],
    p=[0.30, 0.25, 0.20, 0.10, 0.07, 0.05, 0.03], size=N_ACCOUNTS
)
mean_bal_map = {"SAVINGS":50000,"CURRENT":150000,"SALARY":35000,
                "NRE":500000,"NRO":300000,"TRUST":1000000,"SHELL":2000000}
balance = np.array([
    lognorm(mean_bal_map[t], 1.5, 1)[0] for t in acct_type
])

# Dormancy: shell/trust accounts more likely dormant
dormant_prob = np.where(np.isin(acct_type, ["SHELL","TRUST"]), 0.40, 0.08)
acct_status = np.where(
    np.random.rand(N_ACCOUNTS) < dormant_prob, "DORMANT",
    np.random.choice(["ACTIVE","BLOCKED","CLOSED"], p=[0.92,0.05,0.03], size=N_ACCOUNTS)
)

# Number of linked accounts per customer (detected network complexity)
cust_acct_count = pd.Series(acct_cust).value_counts().to_dict()

accounts_df = pd.DataFrame({
    "ACCOUNT_ID"        : acct_ids,
    "CUSTOMER_ID"       : acct_cust,
    "ACCOUNT_TYPE"      : acct_type,
    "ACCOUNT_STATUS"    : acct_status,
    "OPEN_DATE"         : [START_DATE - timedelta(days=int(d))
                            for d in np.random.randint(30, 3650, N_ACCOUNTS)],
    "CURRENT_BALANCE"   : balance,
    "CURRENCY"          : np.random.choice(["INR","USD","AED","SGD","GBP"],
                                            p=[0.55,0.20,0.12,0.08,0.05], size=N_ACCOUNTS),
    "NUM_SIGNATORIES"   : np.random.choice([1,2,3], p=[0.70,0.22,0.08], size=N_ACCOUNTS),
})

print(f"  Accounts:  {accounts_df.shape}")

# ── 1C. TRANSACTIONS ──────────────────────────────────────────
N_BASE    = int(N_TXNS * 0.93)
N_PATTERN = N_TXNS - N_BASE

RAILS = ["UPI","NEFT","RTGS","IMPS","SWIFT","CASH_DEPOSIT","CASH_WITHDRAWAL",
         "POS","CHEQUE","INTERNAL_TRANSFER"]
RAIL_W = [0.28,0.20,0.10,0.12,0.05,0.08,0.06,0.04,0.04,0.03]

NARR_LEGIT = ["SALARY","RENT","VENDOR_PAYMENT","LOAN_EMI","GROCERY",
              "INSURANCE","UTILITY","TRANSFER","INVESTMENT","DIVIDEND"]
NARR_SUSP  = ["REF_MULE","REF_LAYERING","REF_STRUCTURING","REF_RAPID_MVMT",
               "REF_UNKNOWN","N/A"]

HIGH_RISK_CTRY = ["KY","VG","NG","IR","PK"]
acct_arr = np.array(acct_ids)

# Baseline transactions
base_txns = pd.DataFrame({
    "TRANSACTION_ID"  : [f"TXN{i:09d}" for i in range(1, N_BASE+1)],
    "ACCOUNT_ID"      : np.random.choice(acct_arr, size=N_BASE),
    "TXN_TIMESTAMP"   : rand_dates(N_BASE),
    "TXN_TYPE"        : np.random.choice(RAILS, p=RAIL_W, size=N_BASE),
    "TXN_AMOUNT"      : lognorm(15000, 1.4, N_BASE),
    "CURRENCY"        : np.random.choice(["INR","USD","AED"], p=[0.70,0.20,0.10], size=N_BASE),
    "NARRATIVE"       : np.random.choice(NARR_LEGIT, size=N_BASE),
    "BENEFICIARY_COUNTRY": np.random.choice(COUNTRIES, p=COUNTRY_W, size=N_BASE),
    "CHANNEL"         : np.random.choice(["MOBILE","WEB","BRANCH","ATM"], size=N_BASE),
    "IP_ADDRESS"      : [f"192.168.{np.random.randint(0,255)}.{np.random.randint(1,254)}"
                          for _ in range(N_BASE)],
    "IS_TYPOLOGY"     : False,
    "TYPOLOGY_TYPE"   : "NONE",
})

# AML Typology transactions (structuring, layering, mule, rapid movement)
pattern_records = []
typologies = {
    "STRUCTURING"   : {"amount_range":(9000,9990), "rail":"CASH_DEPOSIT",  "narr":"REF_STRUCTURING"},
    "LAYERING"      : {"amount_range":(50000,500000),"rail":"SWIFT",        "narr":"REF_LAYERING"},
    "MULE"          : {"amount_range":(5000,50000), "rail":"UPI",           "narr":"REF_MULE"},
    "RAPID_MOVEMENT": {"amount_range":(20000,200000),"rail":"IMPS",         "narr":"REF_RAPID_MVMT"},
}

for i in range(N_PATTERN):
    t_name, t_cfg = random.choice(list(typologies.items()))
    lo, hi = t_cfg["amount_range"]
    # Layering & SWIFT often involve high-risk destination
    ben_ctry = (random.choice(HIGH_RISK_CTRY)
                if t_name in ["LAYERING","RAPID_MOVEMENT"] and random.random() < 0.6
                else random.choice(COUNTRIES))
    pattern_records.append({
        "TRANSACTION_ID"  : f"TXN{N_BASE+i+1:09d}",
        "ACCOUNT_ID"      : random.choice(acct_arr),
        "TXN_TIMESTAMP"   : datetime.fromtimestamp(random.uniform(start_ts, end_ts)),
        "TXN_TYPE"        : t_cfg["rail"],
        "TXN_AMOUNT"      : round(random.uniform(lo, hi), 2),
        "CURRENCY"        : random.choice(["INR","USD","AED"]),
        "NARRATIVE"       : t_cfg["narr"],
        "BENEFICIARY_COUNTRY": ben_ctry,
        "CHANNEL"         : "MOBILE",
        "IP_ADDRESS"      : f"10.{random.randint(0,9)}.{random.randint(0,255)}.{random.randint(1,254)}",
        "IS_TYPOLOGY"     : True,
        "TYPOLOGY_TYPE"   : t_name,
    })

transactions_df = pd.concat([base_txns, pd.DataFrame(pattern_records)], ignore_index=True)
transactions_df["TXN_TIMESTAMP"] = pd.to_datetime(transactions_df["TXN_TIMESTAMP"])
print(f"  Transactions: {transactions_df.shape}")

# ── 1D. ALERTS ────────────────────────────────────────────────
# Rules that fire alerts
RULES = {
    "R001_HIGH_VALUE_CASH"      : {"threshold": 500000, "col":"TXN_AMOUNT", "op":"gt"},
    "R002_STRUCTURING_SIGNAL"   : {"typology": "STRUCTURING"},
    "R003_LAYERING_SIGNAL"      : {"typology": "LAYERING"},
    "R004_MULE_SIGNAL"          : {"typology": "MULE"},
    "R005_RAPID_MVT"            : {"typology": "RAPID_MOVEMENT"},
    "R006_HIGH_RISK_DEST"       : {"ben_ctry": HIGH_RISK_CTRY},
    "R007_PEP_TRANSACTION"      : {},  # filled below
    "R008_DORMANT_REACTIVATION" : {},
}

alert_rows = []
alert_id_ctr = 1

# Typology alerts — almost all typology txns generate an alert
typo_txns = transactions_df[transactions_df["IS_TYPOLOGY"]]
for _, row in typo_txns.iterrows():
    rule = f"R00{['STRUCTURING','LAYERING','MULE','RAPID_MOVEMENT'].index(row['TYPOLOGY_TYPE'])+2}_SIGNAL" \
           if row['TYPOLOGY_TYPE'] in ['STRUCTURING','LAYERING','MULE','RAPID_MOVEMENT'] else "R001_HIGH_VALUE_CASH"
    # Risk score: typology base + noise
    base_score = {"STRUCTURING":65,"LAYERING":80,"MULE":70,"RAPID_MOVEMENT":75}.get(row['TYPOLOGY_TYPE'],60)
    score = int(min(99, base_score + np.random.randint(-5, 20)))
    alert_rows.append({
        "ALERT_ID"      : f"ALT{alert_id_ctr:08d}",
        "TRANSACTION_ID": row["TRANSACTION_ID"],
        "ACCOUNT_ID"    : row["ACCOUNT_ID"],
        "RULE_TRIGGERED": rule,
        "RISK_SCORE"    : score,
        "ALERT_DATE"    : row["TXN_TIMESTAMP"],
        "IS_TRUE_POS"   : 1,   # ground truth label seed
    })
    alert_id_ctr += 1

# High-value baseline alerts (mostly FP)
hv_txns = transactions_df[
    (~transactions_df["IS_TYPOLOGY"]) & (transactions_df["TXN_AMOUNT"] > 200000)
].sample(frac=0.25, random_state=SEED)
for _, row in hv_txns.iterrows():
    score = int(np.random.randint(40, 75))
    alert_rows.append({
        "ALERT_ID"      : f"ALT{alert_id_ctr:08d}",
        "TRANSACTION_ID": row["TRANSACTION_ID"],
        "ACCOUNT_ID"    : row["ACCOUNT_ID"],
        "RULE_TRIGGERED": "R001_HIGH_VALUE_CASH",
        "RISK_SCORE"    : score,
        "ALERT_DATE"    : row["TXN_TIMESTAMP"],
        "IS_TRUE_POS"   : 0,
    })
    alert_id_ctr += 1

# High-risk destination alerts
hr_txns = transactions_df[
    (~transactions_df["IS_TYPOLOGY"]) &
    (transactions_df["BENEFICIARY_COUNTRY"].isin(HIGH_RISK_CTRY))
].sample(frac=0.15, random_state=SEED)
for _, row in hr_txns.iterrows():
    score = int(np.random.randint(50, 82))
    is_tp = 1 if random.random() < 0.20 else 0  # 20% are genuine
    alert_rows.append({
        "ALERT_ID"      : f"ALT{alert_id_ctr:08d}",
        "TRANSACTION_ID": row["TRANSACTION_ID"],
        "ACCOUNT_ID"    : row["ACCOUNT_ID"],
        "RULE_TRIGGERED": "R006_HIGH_RISK_DEST",
        "RISK_SCORE"    : score,
        "ALERT_DATE"    : row["TXN_TIMESTAMP"],
        "IS_TRUE_POS"   : is_tp,
    })
    alert_id_ctr += 1

alerts_df = pd.DataFrame(alert_rows)
alerts_df["ALERT_DATE"] = pd.to_datetime(alerts_df["ALERT_DATE"])
print(f"  Alerts:    {alerts_df.shape}")
print(f"    True Positive rate in alerts: {alerts_df['IS_TRUE_POS'].mean():.1%}")

# ── 1E. CASES ─────────────────────────────────────────────────
# ~30% of alerts are escalated to cases
case_alerts = alerts_df.sample(frac=0.30, random_state=SEED)
cases_df = pd.DataFrame({
    "CASE_ID"       : [f"CASE{i:07d}" for i in range(1, len(case_alerts)+1)],
    "ALERT_ID"      : case_alerts["ALERT_ID"].values,
    "INVESTIGATOR_ID": np.random.choice(["INV_01","INV_02","INV_03","INV_04","AUTO_SYS"],
                                         size=len(case_alerts)),
    "PRIORITY"      : np.random.choice(["HIGH","MEDIUM","LOW"], p=[0.20,0.45,0.35],
                                        size=len(case_alerts)),
    "CASE_STATUS"   : np.random.choice(
        ["OPEN","CLOSED_FALSE_POSITIVE","CLOSED_SAR_FILED","CLOSED_MONITORING"],
        p=[0.15, 0.65, 0.12, 0.08], size=len(case_alerts)
    ),
    "RESOLUTION_DAYS": np.random.randint(1, 45, size=len(case_alerts)),
})
print(f"  Cases:     {cases_df.shape}")

# ── 1F. INJECT REALISTIC DATA QUALITY ISSUES ─────────────────
def inject_noise(df, cfg):
    d = df.copy()
    for col, ops in cfg.items():
        if col not in d.columns: continue
        if "null_prob" in ops:
            d.loc[np.random.rand(len(d)) < ops["null_prob"], col] = np.nan
        if "case_mess" in ops and d[col].dtype == object:
            mask = np.random.rand(len(d)) < ops["case_mess"]
            d.loc[mask, col] = d.loc[mask, col].apply(
                lambda s: random.choice([str(s).lower(), str(s).upper(), f" {s} "]) if pd.notna(s) else s
            )
    return d

transactions_df = inject_noise(transactions_df, {
    "NARRATIVE"  : {"null_prob": 0.12, "case_mess": 0.15},
    "IP_ADDRESS" : {"null_prob": 0.07},
})
accounts_df = inject_noise(accounts_df, {
    "ACCOUNT_TYPE"  : {"case_mess": 0.10},
    "ACCOUNT_STATUS": {"null_prob": 0.01},
})
customers_df = inject_noise(customers_df, {
    "INCOME_BRACKET": {"null_prob": 0.10},
    "OCCUPATION"    : {"null_prob": 0.05},
})

# Duplicate 0.5% transactions (system retry faults)
dupes = transactions_df.sample(frac=0.005, random_state=SEED)
transactions_df = pd.concat([transactions_df, dupes], ignore_index=True)

# Save raw CSVs (simulating the "upload" step)
# for name, df in [("customers",customers_df),("accounts",accounts_df),
#                   ("transactions",transactions_df),("alerts",alerts_df),("cases",cases_df)]:
#     df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print(f"\n✅ Stage 1 complete. Raw CSVs saved to {OUTPUT_DIR}/")

✅ Imports complete.
Generating synthetic data: 8,000 customers | 12,000 accounts | 120,000 transactions
  Customers: (8000, 12)
  Accounts:  (12000, 8)
  Transactions: (120000, 12)
  Alerts:    (10195, 7)
    True Positive rate in alerts: 85.6%
  Cases:     (3058, 6)

✅ Stage 1 complete. Raw CSVs saved to aml_pipeline_output/


In [7]:
print("\n" + "="*60)
print("STAGE 2 — DATA INGESTION & MERGE")
print("="*60)

# Load from CSV (as the workbench would do after upload)
customers_raw    = pd.read_csv(OUTPUT_DIR / "customers.csv")
accounts_raw     = pd.read_csv(OUTPUT_DIR / "accounts.csv")
transactions_raw = pd.read_csv(OUTPUT_DIR / "transactions.csv")
alerts_raw       = pd.read_csv(OUTPUT_DIR / "alerts.csv")
cases_raw        = pd.read_csv(OUTPUT_DIR / "cases.csv")

for name, df in [("Customers",customers_raw),("Accounts",accounts_raw),
                  ("Transactions",transactions_raw),("Alerts",alerts_raw),("Cases",cases_raw)]:
    print(f"  Loaded {name}: {df.shape[0]:,} rows × {df.shape[1]} cols")

# ── MERGE PIPELINE ────────────────────────────────────────────
print("\nBuilding master feature table...")

master = alerts_raw.copy()
master["ALERT_DATE"] = pd.to_datetime(master["ALERT_DATE"])

# Join case metadata
master = master.merge(
    cases_raw[["ALERT_ID","CASE_ID","PRIORITY","CASE_STATUS","RESOLUTION_DAYS"]],
    on="ALERT_ID", how="left"
)
master["CASE_ID"]         = master["CASE_ID"].fillna("NO_CASE")
master["PRIORITY"]        = master["PRIORITY"].fillna("NONE")
master["RESOLUTION_DAYS"] = master["RESOLUTION_DAYS"].fillna(0)

# Join account features
master = master.merge(
    accounts_raw[["ACCOUNT_ID","CUSTOMER_ID","ACCOUNT_TYPE","ACCOUNT_STATUS",
                  "CURRENT_BALANCE","CURRENCY","NUM_SIGNATORIES"]],
    on="ACCOUNT_ID", how="left"
)

# Join customer features
master = master.merge(
    customers_raw[["CUSTOMER_ID","NATIONALITY","OCCUPATION","INCOME_BRACKET",
                   "CUSTOMER_RISK_RATING","PEP_FLAG","SANCTION_HIT",
                   "ADVERSE_MEDIA_FLAG","KYC_COMPLETENESS_PCT","DAYS_SINCE_KYC",
                   "ONBOARDING_CHANNEL"]],
    on="CUSTOMER_ID", how="left"
)

# ── TRANSACTION AGGREGATES (per account) ─────────────────────
transactions_raw["TXN_TIMESTAMP"] = pd.to_datetime(transactions_raw["TXN_TIMESTAMP"])
transactions_raw["TXN_AMOUNT"]    = pd.to_numeric(transactions_raw["TXN_AMOUNT"], errors="coerce")

txn_agg = transactions_raw.groupby("ACCOUNT_ID").agg(
    total_txn_volume    = ("TXN_AMOUNT", "sum"),
    txn_count           = ("TRANSACTION_ID", "count"),
    avg_txn_amount      = ("TXN_AMOUNT", "mean"),
    max_txn_amount      = ("TXN_AMOUNT", "max"),
    std_txn_amount      = ("TXN_AMOUNT", "std"),
    unique_channels     = ("CHANNEL", "nunique"),
    unique_beneficiary_countries = ("BENEFICIARY_COUNTRY", "nunique"),
    cash_txn_count      = ("TXN_TYPE", lambda x: (x.str.upper().isin(["CASH_DEPOSIT","CASH_WITHDRAWAL"])).sum()),
    swift_txn_count     = ("TXN_TYPE", lambda x: (x.str.upper() == "SWIFT").sum()),
    pct_high_risk_dest  = ("BENEFICIARY_COUNTRY",
                            lambda x: x.isin(HIGH_RISK_CTRY).mean() * 100),
).reset_index()
txn_agg["std_txn_amount"] = txn_agg["std_txn_amount"].fillna(0)
txn_agg["velocity_ratio"] = txn_agg["max_txn_amount"] / (txn_agg["avg_txn_amount"] + 1)

master = master.merge(txn_agg, on="ACCOUNT_ID", how="left")

# Fill numeric NaNs from unmatched accounts
num_fill_cols = ["total_txn_volume","txn_count","avg_txn_amount","max_txn_amount",
                 "std_txn_amount","unique_channels","unique_beneficiary_countries",
                 "cash_txn_count","swift_txn_count","pct_high_risk_dest","velocity_ratio"]
master[num_fill_cols] = master[num_fill_cols].fillna(0)

print(f"Master table shape: {master.shape}")
print(f"Columns: {list(master.columns)}")


STAGE 2 — DATA INGESTION & MERGE
  Loaded Customers: 8,000 rows × 12 cols
  Loaded Accounts: 12,000 rows × 8 cols
  Loaded Transactions: 120,600 rows × 12 cols
  Loaded Alerts: 10,195 rows × 7 cols
  Loaded Cases: 3,058 rows × 6 cols

Building master feature table...
Master table shape: (10195, 38)
Columns: ['ALERT_ID', 'TRANSACTION_ID', 'ACCOUNT_ID', 'RULE_TRIGGERED', 'RISK_SCORE', 'ALERT_DATE', 'IS_TRUE_POS', 'CASE_ID', 'PRIORITY', 'CASE_STATUS', 'RESOLUTION_DAYS', 'CUSTOMER_ID', 'ACCOUNT_TYPE', 'ACCOUNT_STATUS', 'CURRENT_BALANCE', 'CURRENCY', 'NUM_SIGNATORIES', 'NATIONALITY', 'OCCUPATION', 'INCOME_BRACKET', 'CUSTOMER_RISK_RATING', 'PEP_FLAG', 'SANCTION_HIT', 'ADVERSE_MEDIA_FLAG', 'KYC_COMPLETENESS_PCT', 'DAYS_SINCE_KYC', 'ONBOARDING_CHANNEL', 'total_txn_volume', 'txn_count', 'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'unique_channels', 'unique_beneficiary_countries', 'cash_txn_count', 'swift_txn_count', 'pct_high_risk_dest', 'velocity_ratio']


In [9]:
master.head()

,ALERT_ID,TRANSACTION_ID,ACCOUNT_ID,RULE_TRIGGERED,RISK_SCORE,ALERT_DATE,IS_TRUE_POS,CASE_ID,PRIORITY,CASE_STATUS,...,txn_count,avg_txn_amount,max_txn_amount,std_txn_amount,unique_channels,unique_beneficiary_countries,cash_txn_count,swift_txn_count,pct_high_risk_dest,velocity_ratio
0,ALT00000001,TXN000111601,ACCT100004507,R002_SIGNAL,70,2022-08-12 02:18:55.769613,1,CASE0000791,LOW,CLOSED_FALSE_POSITIVE,...,12,16251.210833,96215.50,27548.585562,4,6,4,0,16.666667,5.920148
1,ALT00000002,TXN000111602,ACCT100003583,R005_SIGNAL,84,2022-07-31 22:53:51.983142,1,NO_CASE,NONE,NaN,...,10,15217.511000,128363.37,39802.018085,4,5,0,1,10.000000,8.434687
2,ALT00000003,TXN000111603,ACCT100002616,R005_SIGNAL,82,2023-09-29 00:07:11.035281,1,NO_CASE,NONE,NaN,...,10,35869.199000,149003.55,51516.462207,4,5,4,0,10.000000,4.153965
3,ALT00000004,TXN000111604,ACCT100001585,R002_SIGNAL,60,2022-11-24 00:43:30.629697,1,CASE0000664,MEDIUM,OPEN,...,11,5170.277273,19318.18,5517.316891,4,5,2,0,0.000000,3.735669
4,ALT00000005,TXN000111605,ACCT100001292,R002_SIGNAL,78,2023-05-18 21:48:58.903768,1,NO_CASE,NONE,NaN,...,10,21528.411000,164227.03,50348.171119,4,6,4,0,30.000000,7.628032
